<h1 align="center"> AI Product Recommendation Assistant</h1>

<p>
This project uses Large Language Models (LLMs), Pydantic validation, and a product dataset to recommend products based on natural language queries.
</p>

<h3>Workflow</h3>

<pre>
┌──────────────┐
│  User Query  │
└──────┬───────┘
       │
       ▼
┌─────────────────────────┐
│ Filter Extraction (LLM) │
└──────┬──────────────────┘
       │
       ▼
┌─────────────────────┐
│ Pydantic Validation │
└──────┬──────────────┘
       │
       ▼
┌─────────────────────┐
│ Product Search Tool │
└──────┬──────────────┘
       │
       ▼
┌─────────────────────┐
│ Recommendation LLM  │
└──────┬──────────────┘
       │
       ▼
┌─────────────────────┐
│   Final Response    │
└─────────────────────┘
</pre>


In [6]:
import json

from openai import OpenAI

from pydantic import BaseModel

from enum import Enum

## Dataset Loading

Load the product catalog which contains information about products such as category, brand, price, rating and description.

In [7]:
with open("products.json", "r") as f:
    products = json.load(f)

print(f"Total Products: {len(products)}")

Total Products: 100


## Product Categories

Enum ensures only valid categories can be used inside the system.

In [8]:
class Category(str, Enum):
    CAMERAS = "cameras"
    SPEAKERS = "speakers"
    WEBCAMS = "webcams"
    TABLETS = "tablets"
    SMARTPHONES = "smartphones"
    HEADPHONES = "headphones"
    KEYBOARDS = "keyboards"
    MONITORS = "monitors"
    LAPTOPS = "laptops"
    SMARTWATCHES = "smartwatches"

## Search Request Schema

Pydantic is used to validate the filters extracted by the LLM before they are passed to the search tool.

In [9]:
class ProductSearchRequest(BaseModel):

    category: Category | None = None
    brand: str | None = None
    max_price: int | None = None
    min_rating: float | None = None

## LLM Configuration

Connect to OpenRouter and initialize the language model.

In [10]:
from openai import OpenAI

client = OpenAI(
    api_key="api key here",
    base_url="https://openrouter.ai/api/v1"
)

## Filter Extraction

The LLM converts a natural language query into structured filters.

Example:

"Recommend a laptop under 90000"

↓

{
  "category": "laptops",
  "max_price": 90000
}

In [11]:
import json

def extract_filters(query):

    response = client.chat.completions.create(
        model="nex-agi/nex-n2-pro:free",
        max_tokens=200,
        messages=[
            {
                "role": "system",
                "content": """
                You are a product filter extraction assistant.

                Extract:
                - category
                - brand
                - max_price
                - min_rating

                Valid categories:
                - cameras
                - speakers
                - webcams
                - tablets
                - smartphones
                - headphones
                - keyboards
                - monitors
                - laptops
                - smartwatches

                IMPORTANT:
                - If user says laptop, return "laptops"
                - If user says phone, return "smartphones"
                - If user says watch, return "smartwatches"
                - Use ONLY the exact category values listed above

                Return ONLY valid JSON.
                """
            },
            {
                "role": "user",
                "content": query
            }
        ]
    )

    data = json.loads(
        response.choices[0].message.content
    )

    return ProductSearchRequest(**data)

## Product Search Tool

This tool searches the dataset using the filters extracted by the LLM.

In [12]:
def search_products(filters):

    results = []

    for product in products:

        if (
            filters.category and
            product["category"].lower()
            != filters.category.value.lower()
        ):
            continue

        if (
            filters.brand and
            product["brand"].lower()
            != filters.brand.lower()
        ):
            continue

        if (
            filters.max_price and
            product["attributes"]["price"]
            > filters.max_price
        ):
            continue

        if (
            filters.min_rating and
            product["rating"]
            < filters.min_rating
        ):
            continue

        results.append(product)

    return results

## Product Recommendation

The retrieved products are sent to the LLM which ranks and explains the best options.

In [13]:
import json

def recommend_products(query, products):

    products_json = json.dumps(
        products,
        indent=2
    )

    response = client.chat.completions.create(
        model="nex-agi/nex-n2-pro:free",
        max_tokens=1500,
        messages=[
            {
                "role": "system",
                "content": """
                You are a product recommendation expert.

                Recommend the best products from the provided list.

                Rank the products from best to worst.

                For each product provide:
                - Product Name
                - Why it is suitable
                - Pros
                - Cons

                Only use the products provided.
                Do not invent products.

                Keep the answer concise.
                """
            },
            {
                "role": "user",
                "content": f"""
                User Query:
                {query}

                Products:
                {products_json}
                """
            }
        ]
    )

    return response.choices[0].message.content

## Query Moderation

Simple safety layer to block harmful or unsupported queries.

In [14]:
def moderate_query(query):

    blocked_words = [
        "hack",
        "password",
        "exploit",
        "bypass",
        "illegal"
    ]

    query = query.lower()

    for word in blocked_words:
        if word in query:
            return False

    return True

## End-to-End Pipeline

Combines all components into a single AI assistant.

In [15]:
def product_assistant(query):

    if not moderate_query(query):
        return "Sorry, I cannot help with that request."

    filters = extract_filters(query)

    results = search_products(filters)

    if not results:
        return "No matching products found."

    return recommend_products(query, results)

## Example Query

In [16]:
print(
    product_assistant(
        "Recommend a laptop under 90000 for programming"
    )
)

All listed laptops are under ₹90,000 and have 16GB RAM, which is good for programming.

| Rank | Product | Why it is suitable | Pros | Cons |
|---|---|---|---|---|
| 1 | **Lenovo LOQ 15** | Best overall for programming, especially if you do heavy development, AI/ML, Android development, or creative workloads. | RTX 4050 GPU, 16GB RAM, strong performance, good thermals | Heavy at 2.4 kg, only 5 hours battery |
| 2 | **HP Pavilion Plus 14** | Best portable option for coding, students, and office productivity. | Lightweight 1.4 kg, 16GB RAM, 9 hours battery, slim design | Intel Iris Xe graphics, less powerful for GPU-heavy tasks |
| 3 | **ASUS Vivobook S15** | Good balanced laptop for general programming, web development, and study/work use. | 16GB RAM, 10 hours battery, affordable, comfortable for daily use | Intel Iris Xe graphics, not ideal for heavy gaming or GPU workloads |
| 4 | **Acer Nitro V** | Budget-friendly option with dedicated graphics for development and light gaming. | 16G

## Evaluation

Testing the assistant on multiple user queries.

In [17]:
test_queries = [
    "Recommend a laptop under 90000",
    "Best Sony camera",
    "Headphones under 10000",
    "Tablet under 50000",
    "Monitor under 20000"
]

for query in test_queries:

    print("=" * 50)
    print("Query:", query)

    print(
        product_assistant(query)
    )

Query: Recommend a laptop under 90000
All options are under ₹90,000. Ranked best to worst:

### 1. Lenovo LOQ 15 — ₹82,999
**Why suitable:** Best overall choice under ₹90k, especially for gaming, programming, and creative workloads.  
**Pros:** RTX 4050, 16GB RAM, highest rating 4.2, strong performance.  
**Cons:** Heavier at 2.4 kg, battery life only 5 hours.

### 2. HP Pavilion Plus 14 — ₹72,999
**Why suitable:** Best for students, office users, and portability-focused buyers.  
**Pros:** Lightweight at 1.4 kg, 9-hour battery, 16GB RAM, good display quality.  
**Cons:** Intel Iris Xe GPU is not ideal for gaming or heavy creative work, costs more than Vivobook S15.

### 3. ASUS Vivobook S15 — ₹64,999
**Why suitable:** Good everyday laptop with strong battery life and comfortable use for students/professionals.  
**Pros:** Longest battery at 10 hours, 16GB RAM, affordable, relatively light at 1.6 kg.  
**Cons:** Integrated Intel Iris Xe GPU, not suitable for demanding gaming or creativ

<h2>Conclusion</h2>

<p>
This project demonstrates how Large Language Models (LLMs) can be integrated with structured data and validation frameworks to build an intelligent product recommendation assistant.

The system converts natural language queries into structured filters, validates them using Pydantic, searches a product dataset, and generates personalized recommendations using an LLM.

This approach combines AI reasoning with traditional software engineering principles to create reliable and explainable recommendations.
</p>